# Fine-tuning — Arandu Nano 1.0

Treina um modelo proprio sobre o **Llama-3.2-1B-Instruct** (open source) com os seus dados, usando **Unsloth** (LoRA/QLoRA). Ao final, exporta um **.gguf Q4_K_M** pronto para rodar no llamafile na USB.

## Como usar
1. Abra no Google Colab (gratis).
2. Menu **Ambiente de execucao -> Alterar tipo** -> selecione **GPU (T4)**.
3. Rode as celulas em ordem (Shift+Enter).
4. Faca upload do seu `dataset.jsonl` quando pedido (use o `dataset_exemplo.jsonl` como modelo e va aumentando).
5. No fim, baixe `arandu-nano-1.0-Q4_K_M.gguf`, coloque em `D:\IAPortatil\` e ajuste o `modelo.txt`.

In [ ]:
# 1) Instala o Unsloth (treino rapido, baixa VRAM)
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2) Carrega o modelo base em 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,            # detecta automaticamente
    load_in_4bit = True,     # QLoRA: cabe folgado na T4
)

In [ ]:
# 3) Adiciona os adaptadores LoRA (so eles sao treinados -> rapido e leve)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4) Upload do dataset.jsonl (formato: uma linha por exemplo, com 'messages')
from google.colab import files
print("Selecione o seu dataset.jsonl (modelo: dataset_exemplo.jsonl)")
up = files.upload()
DATA = list(up.keys())[0]
print("Arquivo:", DATA)

In [ ]:
# 5) Prepara os dados com o chat template do Llama 3
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")

def formatar(ex):
    txt = tokenizer.apply_chat_template(ex["messages"], tokenize=False,
                                        add_generation_prompt=False)
    return {"text": txt}

dataset = load_dataset("json", data_files=DATA, split="train")
dataset = dataset.map(formatar)
print("Exemplos:", len(dataset))
print(dataset[0]["text"][:600])

In [ ]:
# 6) Treina (LoRA). Ajuste num_train_epochs conforme o tamanho do dataset.
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,        # poucos dados: 3-5 epocas. Muitos dados: use max_steps.
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "saida",
        save_strategy = "no",      # evita o bug de pickle ao salvar checkpoint
        report_to = "none",
    ),
)
trainer.train()

In [ ]:
# 7) Teste rapido do modelo treinado
FastLanguageModel.for_inference(model)
msgs = [{"role":"user","content":"Quem e voce e o que voce faz?"}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True,
            add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.5)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
# 8) Exporta para GGUF Q4_K_M (mescla o LoRA) e baixa
model.save_pretrained_gguf("arandu-nano-1.0", tokenizer,
                           quantization_method = "q4_k_m")

import glob
from google.colab import files
# o Unsloth salva em uma subpasta (ex.: arandu-nano-1.0_gguf/...Q4_K_M.gguf)
ggufs = glob.glob("**/*.gguf", recursive=True)
print("Gerados:", ggufs)
q4 = [f for f in ggufs if "Q4_K_M" in f or "q4_k_m" in f]
alvo = q4[0] if q4 else ggufs[0]
print("Baixando:", alvo)
files.download(alvo)  # renomeie p/ arandu-nano-1.0-Q4_K_M.gguf na USB